In [9]:
import pandas as pd
import numpy as np

#1. LOAD & CLEAN DATA (numeric fix + fallback CP calculation)

In [10]:
def load_data(path):
    df = pd.read_csv(path)

    # Rename Japanese columns → English standard
    rename_map = {
        "所在地": "location",
        "管理費_共益費": "management_fee",
        "向き": "direction",
        "階": "floor_raw",
        "条件": "conditions",
    }
    df.rename(columns=rename_map, inplace=True)

    # Parse floor from format "3階/10階建"
    if "floor_raw" in df.columns:
        df["floor"] = (
            df["floor_raw"]
            .astype(str)
            .str.extract(r"(\d+)", expand=False)
            .astype(float)
        )
    else:
        df["floor"] = 0

    # Parse pet_allowed from Japanese
    # Strict Option A:
    # "ペット相談可", "ペット可" → allowed
    # everything else → not allowed
    if "conditions" in df.columns:
        df["pet_allowed"] = df["conditions"].astype(str).apply(
            lambda x: 1 if ("ペット" in x and "不可" not in x) and ("可" in x) else 0
        )
    else:
        df["pet_allowed"] = 0

    # Ensure numeric fields cleaned
    numeric_cols = [
        "rent", "management_fee", "area_m2",
        "building_age", "walking_distance_to_station",
        "敷金", "礼金"
    ]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = (
                df[col].astype(str)
                .str.replace(",", "")
                .str.replace("¥", "")
                .str.extract(r"([-]?\d+\.?\d*)", expand=False)
            )
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Fill NaN only for numeric columns (pandas>=2 string dtype safe)
    num_cols = df.select_dtypes(include=["number"]).columns
    df[num_cols] = df[num_cols].fillna(0)

    # Compute cost_performance (fallback-safe)
    if "management_fee" not in df.columns:
        df["management_fee"] = 0

    df["total_cost_temp"] = df["rent"] + df["management_fee"]
    df["total_cost_temp"] = df["total_cost_temp"].replace(0, 1)
    df["cost_performance"] = df["area_m2"] / df["total_cost_temp"]

    df.drop(columns=["total_cost_temp"], inplace=True)

    return df


#2. PRIORITIZED FILTER ENGINE

In [11]:
SOUTH_KEYWORDS = ["南", "south"]

def apply_filters(df, filters):
    if df is None or not isinstance(df, pd.DataFrame):
        return pd.DataFrame()

    filtered = df.copy()

    rules = {
        "max_rent": lambda d, v: d[d["rent"] <= v] if "rent" in d.columns else d,
        "max_distance": lambda d, v: d[d["walking_distance_to_station"] <= v]
            if "walking_distance_to_station" in d.columns else d,
        "min_area": lambda d, v: d[d["area_m2"] >= v] if "area_m2" in d.columns else d,
        "max_age": lambda d, v: d[d["building_age"] <= v]
            if "building_age" in d.columns else d,

        # Japanese location
        "area": lambda d, v: d[d["location"].fillna("").str.contains(v, case=False)]
            if "location" in d.columns else d,

        # Japanese direction (南向き)
        "south_facing": lambda d, v: d[
            d["direction"].fillna("").str.contains("|".join(SOUTH_KEYWORDS), case=False)
        ] if "direction" in d.columns else d,

        "allow_pets": lambda d, v: d[d["pet_allowed"] == int(v)]
            if "pet_allowed" in d.columns else d,

        "floor": lambda d, v: d[d["floor"] == v] if "floor" in d.columns else d,
    }

    for key in filters:
        if key in rules and filters[key] is not None:
            try:
                new_df = rules[key](filtered, filters[key])
                if isinstance(new_df, pd.DataFrame):
                    filtered = new_df
            except Exception:
                continue

    return filtered

#3. FALLBACK ENGINE — FIND CLOSEST MATCH

In [12]:
PRIORITY = ["max_rent", "max_distance", "min_area", "max_age"]

def fallback_filter(df, filters):
    for k in PRIORITY:
        if k in filters and filters[k] is not None:
            temp_f = {k: filters[k]}
            result = apply_filters(df, temp_f)
            if len(result) > 0:
                return result, f"Matched only on: {k}"

    return df, "No match. Showing general best candidates."


#4. SCORE SYSTEM – AI-LIKE RANKING

In [13]:
def compute_scores(df, income, persons):
    df = df.copy()

    # Ideal size formula:
    ideal_area = persons * 20  # good baseline for Japan

    # Normalize useful fields
    def norm(col):
        mn, mx = df[col].min(), df[col].max()
        if mx - mn == 0:
            return df[col] * 0
        return (df[col] - mn) / (mx - mn)

    df["norm_rent"] = norm("rent")
    df["norm_distance"] = norm("walking_distance_to_station")
    df["norm_cp"] = norm("cost_performance")

    df["budget_score"] = 1 - df["norm_rent"]
    df["distance_score"] = 1 - df["norm_distance"]

    df["size_score"] = np.exp(-abs(df["area_m2"] - ideal_area) / ideal_area)

    df["age_score"] = 1 - (df["building_age"] / (df["building_age"].max() + 1))

    df["final_score"] = (
        df["budget_score"] * 0.30 +
        df["distance_score"] * 0.25 +
        df["size_score"] * 0.20 +
        df["age_score"] * 0.10 +
        df["norm_cp"] * 0.15
    )

    return df.sort_values("final_score", ascending=False)



#5. EXPLANATION ENGINE

In [14]:
def generate_explanation(top5, fallback_msg, full_df):
    best = top5.iloc[0]

    explanation = []

    explanation.append(f"🎯 **Top match found!** ({fallback_msg})")
    explanation.append(f"- Rent: ¥{int(best['rent'])}")
    explanation.append(f"- Area: {best['area_m2']} m²")
    explanation.append(f"- Distance: {best['walking_distance_to_station']} min to station")
    explanation.append(f"- Direction: {best['direction']}")

    if best["rent"] < full_df["rent"].median():
        explanation.append("💰 **Great deal compared to regional market price!**")

    explanation.append("\n### Why this property is recommended:")
    explanation.append("• Good cost performance for its size")
    explanation.append("• Balanced rent vs quality")
    explanation.append("• Reasonable walking distance")

    return "\n".join(explanation)



#6. RECOMMEND FUNCTION

In [15]:
def recommend(df, income, persons, **filters):
    if df is None or not isinstance(df, pd.DataFrame):
        raise ValueError("recommend() received invalid dataframe.")

    filtered = apply_filters(df, filters)

    if len(filtered) == 0:
        filtered, fb_msg = fallback_filter(df, filters)
    else:
        fb_msg = "Matched all criteria"

    ranked = compute_scores(filtered, income, persons)
    top5 = ranked.head(5)

    explanation = generate_explanation(top5, fb_msg, df)

    return top5, explanation

#7. MAIN FOR TESTING

In [16]:
if __name__ == "__main__":
    df = load_data(r".\data\yodogawa_feature_eng.csv")

    top5, exp = recommend(
        df,
        income=300000,
        persons=2,
        max_rent=120000,
        max_distance=15,
        min_area=25,
        max_age=20,
        south_facing=True,
        allow_pets=False,
    )

    print(top5)
    print(exp)

        id  phone_number  rent  management_fee  deposit  key_money  \
1331  1394  06-6100-7788   6.0          3400.0      0.0        0.0   
1332  1395  06-6306-6061   6.0          3700.0      0.0        6.0   
1333  1397  06-6306-6061   6.2          3700.0      0.0        6.0   
646    656  06-6868-9571   6.3          5000.0      0.0       12.6   
648    658  06-6575-9711   6.3          5000.0      0.0       12.6   

      guarantee_deposit  deposit_amortization        location  \
1331                0.0                   0.0  大阪府大阪市淀川区三津屋北１   
1332                0.0                   0.0  大阪府大阪市淀川区三津屋北１   
1333                0.0                   0.0  大阪府大阪市淀川区三津屋北１   
646                 0.0                   0.0  大阪府大阪市淀川区三津屋中１   
648                 0.0                   0.0  大阪府大阪市淀川区三津屋中１   

      walking_distance_to_station  ... pet_allowed  cost_performance  \
1331                            4  ...           0          0.008538   
1332                            5  ...      